# Silent Film Restoration — Colab Bootstrap

Thin notebook. All logic lives in the `pipeline/` package.

1. Mount Google Drive at `/content/drive/MyDrive/silent-film-restoration/lanka-dahan/`.
2. Clone the repo into `/content/Silent-Film-Restoration/`.
3. Install deps.
4. Run stages. Outputs land in Drive (persistent across session deaths).

### Keep-alive JS (paste in browser console to prevent 90-min idle disconnect)
Client-side only. Does NOT bypass the 12-hour hard cap.
```js
function KeepAlive(){console.log('keep-alive ping');document.querySelector('colab-connect-button').shadowRoot.querySelector('#connect').click();}
setInterval(KeepAlive, 60000);
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib
PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/silent-film-restoration/lanka-dahan')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['SILENT_FILM_ENV'] = 'colab'
print('Drive root:', PROJECT_ROOT)

In [ ]:
%cd /content
![ -d Silent-Film-Restoration ] || git clone https://github.com/utsavbansal93/Silent-Film-Restoration.git
%cd /content/Silent-Film-Restoration
!git pull

In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q -e '.[dev]'

In [ ]:
# Sync source file from Drive if present; otherwise fetch from Wikimedia to Drive.
DRIVE_SRC = PROJECT_ROOT / 'source' / 'lanka_dahan_1917.webm'
LOCAL_SRC = pathlib.Path('/content/Silent-Film-Restoration/source/lanka_dahan_1917.webm')
LOCAL_SRC.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_SRC.exists():
    !ln -sf "$DRIVE_SRC" "$LOCAL_SRC"
else:
    !bash scripts/fetch_source.sh
    !mkdir -p "$(dirname '$DRIVE_SRC')" && cp '$LOCAL_SRC' '$DRIVE_SRC'
!bash scripts/fetch_east_model.sh

In [ ]:
# Determinism check: run S00 on fixture, compare to M3 output hashes.
# Expected bar: byte-identical PNGs. Any drift is a bug.
!bash scripts/regenerate_fixture.sh
!python -m pipeline.stages.s00_ingest --config configs/test_30sec.yaml

## Kaggle fallback (toggle on when Colab quota runs out)

Uncomment and run these cells instead of the Colab mount/setup above. Paths use `/kaggle/working/` and `SILENT_FILM_ENV=kaggle`.

```python
# import os, pathlib
# PROJECT_ROOT = pathlib.Path('/kaggle/working/silent-film-restoration/lanka-dahan')
# PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
# os.environ['SILENT_FILM_ENV'] = 'kaggle'
# !git clone https://github.com/utsavbansal93/Silent-Film-Restoration.git /kaggle/working/Silent-Film-Restoration
# %cd /kaggle/working/Silent-Film-Restoration
# !pip install -q -e '.[dev]'
# # Attach the Wikimedia WebM as a Kaggle dataset and symlink it into source/.
# !bash scripts/fetch_east_model.sh
# !python -m pipeline.run --config configs/modern_smooth.yaml
```